# Visualization for Experiment 1: Exact ODE solution

Run all cells to load the named pilot run and draw the main-text phase portrait.
There is no date or run ID to choose. Generate data from the repository root:

```bash
uv run muon-ode                     # five Table 1 dynamics
uv run muon-ode dynamics=all        # all ten, saved separately
```

Each successful rerun updates its named location. Older snapshots remain intact;
this notebook resolves the location once so its curves always come from one run.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter

from ode.observables import errors, learning_times, margins, probabilities
from ode.results import condition_label, condition_names, load_run

# Works when Jupyter starts in the repository root or notebooks/.
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src/ode").is_dir()
)

EXPERIMENT = "pilot"
DYNAMICS = "main_text"     # "main_text" (5) or "all" (10): which named run to load
PLOT_DYNAMICS = None       # None follows DYNAMICS; "main_text" selects 5 from an all-ten run
PAIR = None               # pilot picks its saved pair; set (S, R) for a cardinality run
OUTPUT_ROOT = PROJECT_ROOT / "runs/experiment1"

# Explicit sizes keep rerunning this cell idempotent (twice the previous fonts).
plt.rcParams.update({
    "font.size": 20,
    "axes.labelsize": 20,
    "axes.titlesize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 18,
})


## Experiment 1A: Pilot

One curve per dynamic, with $p_S$ on the horizontal axis and $p_R$ on the vertical
axis. Subject/relation counts and initialization are read from the saved run.
The diagonal indicates equal component probabilities.

`experiment.delta` is the **stopping error**: a run stops when both components
reach at least $1-\delta$ (or reaches its time cap). It is not a list of plot
thresholds. The pilot default is $\delta=0.001$. Learning times at other errors can
be calculated later with `observables.learning_times`; these use interpolation
of saved trajectories, and thresholds beyond the recorded trajectory are unreached.


In [ ]:
recorded = load_run(
    OUTPUT_ROOT,
    experiment=EXPERIMENT,
    dynamics=DYNAMICS,
    plot_dynamics=PLOT_DYNAMICS,
    pair=PAIR,
)
first_case = next(iter(recorded.cases.values()))
problem = first_case.config.problem
subjects, relations = problem.subjects, problem.relations
print(f"Loaded {EXPERIMENT}/{DYNAMICS}: S={subjects}, R={relations}")
print(f"Stopping delta: {recorded.config.experiment.delta}")
print("Curves:", ", ".join(condition_label(name) for name in recorded.cases))

# Stable export paths, separate from immutable simulation snapshots.
FIGURE_DIR = OUTPUT_ROOT / "figures" / EXPERIMENT / DYNAMICS
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_STEM = f"{PLOT_DYNAMICS or DYNAMICS}_S{subjects}_R{relations}"


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5), constrained_layout=True)
# Keep each dynamic's color fixed when switching between main text and appendix.
colors = dict(zip(condition_names("all"), plt.get_cmap("tab10").colors))
for condition, case in recorded.cases.items():
    p_s, p_r, _ = probabilities(case.state, subjects, relations)
    ax.plot(
        p_s, p_r,
        label=condition_label(condition),
        color=colors[condition],
        linewidth=2,
        linestyle="-" if condition in {"gf", "spectral"} else "--",
    )
ax.plot([0, 1], [0, 1], color="0.75", linewidth=1, linestyle=":", zorder=0)
ax.set(xlabel=r"$p_S$", ylabel=r"$p_R$", xlim=(0, 1), ylim=(0, 1))
ax.set_aspect("equal", adjustable="box")
ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1])
ax.set_title(f"S = {subjects}, R = {relations}")
ax.legend(frameon=False, fontsize=18, loc="center right", bbox_to_anchor=(0.98, 1 / 3))
pdf_path = FIGURE_DIR / f"{FIGURE_STEM}_probabilities.pdf"
fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
print(f"Saved PDF: {pdf_path}")
plt.show()


### Log-odds phase portrait

Plot $q_S$ against $q_R$, using the identities
$q_S=\log(p_S/(1-p_S))$ and $q_R=\log(p_R/(1-p_R))$.
The margins are computed directly from alpha/beta to avoid numerical problems
when a probability rounds to one. Both plots use equal units on the two axes;
the log-odds plot also uses identical limits on both axes.


In [ ]:
fig_q, ax_q = plt.subplots(figsize=(6.5, 6.5), constrained_layout=True)
q_curves = {
    condition: margins(case.state, subjects, relations)
    for condition, case in recorded.cases.items()
}
for condition, (q_s, q_r) in q_curves.items():
    ax_q.plot(
        q_s, q_r,
        label=condition_label(condition),
        color=colors[condition],
        linewidth=2,
        linestyle="-" if condition in {"gf", "spectral"} else "--",
    )
q_min = min(float(values.min()) for curve in q_curves.values() for values in curve)
q_max = max(float(values.max()) for curve in q_curves.values() for values in curve)
padding = 0.05 * max(q_max - q_min, 1.0)
q_limits = (q_min - padding, q_max + padding)
ax_q.plot(q_limits, q_limits, color="0.75", linewidth=1, linestyle=":", zorder=0)
ax_q.set(
    xlabel=r"$\log\!\left(\frac{p_S}{1-p_S}\right)$",
    ylabel=r"$\log\!\left(\frac{p_R}{1-p_R}\right)$",
    xlim=q_limits,
    ylim=q_limits,
)
ax_q.set_aspect("equal", adjustable="box")
ax_q.set_title(f"S = {subjects}, R = {relations}")
ax_q.legend(frameon=False, fontsize=18)
pdf_path = FIGURE_DIR / f"{FIGURE_STEM}_log_odds.pdf"
fig_q.savefig(pdf_path, format="pdf", bbox_inches="tight")
print(f"Saved PDF: {pdf_path}")
plt.show()


### Component error versus time

Separate panels show GF and spectral GF, each with $1-p_S$ and $1-p_R$ on a
logarithmic y-axis. Horizontal lines mark 90%, 99%, and 99.9% accuracy; vertical
lines and intersection markers use the corresponding subject/relation colors.
Learning times are interpolated from saved margins. The pilot now stops at
$\delta=0.001$ so every requested crossing is recorded.


In [ ]:
ACCURACIES = (0.9, 0.99, 0.999)
component_colors = ("tab:blue", "tab:orange")
error_figures = {}
for condition in ("gf", "spectral"):
    if condition not in recorded.cases:
        raise ValueError(f"The selected run must include {condition} for the error plots.")
    case = recorded.cases[condition]
    crossings = {
        accuracy: learning_times(case.time, case.state, subjects, relations, 1 - accuracy)[:2]
        for accuracy in ACCURACIES
    }
    if not all(np.isfinite(times).all() for times in crossings.values()):
        raise ValueError(
            "The saved run does not reach every requested accuracy. Rerun "
            f"uv run muon-ode experiment={EXPERIMENT} dynamics={DYNAMICS} "
            "experiment.delta=0.001, then reload the notebook."
        )
    error_s, error_r = errors(case.state, subjects, relations)
    fig_error, ax_error = plt.subplots(figsize=(10, 7), constrained_layout=True)
    for values, label, color in zip(
        (error_s, error_r), (r"$1-p_S$", r"$1-p_R$"), component_colors, strict=True
    ):
        ax_error.semilogy(case.time, values, label=label, color=color, linewidth=2)
    for accuracy, times in crossings.items():
        target_error = 1 - accuracy
        ax_error.axhline(target_error, color="0.6", linestyle=":", linewidth=1)
        ax_error.text(
            1.01, target_error, f"{100 * accuracy:g}%",
            transform=ax_error.get_yaxis_transform(), va="center", fontsize=20,
        )
        for crossing, color in zip(times, component_colors, strict=True):
            ax_error.axvline(crossing, color=color, linestyle="--", linewidth=1, alpha=0.7)
            ax_error.scatter(crossing, target_error, color=color, s=55, zorder=5)
    ax_error.set(
        xlabel=r"$t$", ylabel="Component error",
        title=f"{condition_label(condition)}: S = {subjects}, R = {relations}",
        xlim=(0, case.time[-1] * 1.04),
    )
    ax_error.legend(frameon=False)
    pdf_path = FIGURE_DIR / f"{FIGURE_STEM}_{condition}_errors.pdf"
    fig_error.savefig(pdf_path, format="pdf", bbox_inches="tight")
    print(f"Saved PDF: {pdf_path}")
    error_figures[condition] = (fig_error, ax_error)
    plt.show()


### Other selections

- For all ten curves, run `uv run muon-ode dynamics=all` and set `DYNAMICS = "all"`.
- To show the five main-text curves from that same all-ten run, also set
  `PLOT_DYNAMICS = "main_text"`.
- To change the stopping error, run e.g.
  `uv run muon-ode experiment.delta=0.001`, then rerun this notebook.
- For a precision run, use `uv run muon-ode experiment=precision` and set
  `EXPERIMENT = "precision"`. Its single stopping error is $10^{-6}$.

The runner writes raw time/alpha/beta data and metadata only. All probabilities
and all figures above are generated here; no separate plotting module is needed.

### PDF exports

Running the plotting cells also saves vector PDFs under
`runs/experiment1/figures/<experiment>/<run dynamics>/`.
Filenames include the plotted dynamics, S, R, and `probabilities`, `log_odds`, `gf_errors`, or `spectral_errors`.
Rerunning the same selection replaces its exported PDFs. Exports are separate
from the raw-data snapshots and remain outside Git under `runs/`.


## Appendix: learning-time separation and softmax saturation

These four plots compare **GF and spectral GF only**. Generate their named runs:

```bash
uv run muon-ode experiment=ratio_fixed_s dynamics=gf_spectral
uv run muon-ode experiment=ratio_fixed_r dynamics=gf_spectral
uv run muon-ode experiment=appendix_precision dynamics=gf_spectral
```

The ratio is $T_S^\delta/T_R^\delta$, where each learning time is the first time
its component probability reaches $1-\delta$. The two cardinality slices use
$\delta=0.1$ (90%); change `experiment.delta` in both runs to use another threshold.
The two separate GF and spectral GF learning-time plots use $(S,R)=(64,16)$ and errors from $10^{-6}$ to $0.5$.
It measures both components using one saved trajectory per flow, stopped at
$10^{-6}$. Learning times at intermediate errors are interpolated from the
saved margins. All four plots use logarithmic scales on both axes.

These finite-size results test the theoretical trends, including transients and
logarithmic factors; they do not assume the ratios equal the asymptotic formulas.


In [ ]:
APPENDIX_DYNAMICS = "gf_spectral"
APPENDIX_FIGURE_DIR = OUTPUT_ROOT / "figures" / "appendix" / APPENDIX_DYNAMICS
APPENDIX_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
flow_colors = {"gf": "tab:blue", "spectral": "tab:orange"}
ratio_results = {}
ratio_figures = {}

for experiment_name, fixed_axis, fixed_value, varying_axis in (
    ("ratio_fixed_s", "S", 1024, "R"),
    ("ratio_fixed_r", "R", 4, "S"),
):
    # Load once to pin the snapshot; all subsequent pairs use that exact snapshot.
    first_pair = (1024, 4) if fixed_axis == "S" else (4, 4)
    suite = load_run(
        OUTPUT_ROOT, experiment=experiment_name, dynamics=APPENDIX_DYNAMICS, pair=first_pair
    )
    ratio_delta = float(suite.config.experiment.delta)
    pairs = [tuple(pair) for pair in suite.config.experiment.pairs]
    variable_index = 1 if varying_axis == "R" else 0
    pairs.sort(key=lambda pair: pair[variable_index])
    if any(pair[1 - variable_index] != fixed_value for pair in pairs):
        raise ValueError(f"{experiment_name} must fix {fixed_axis}={fixed_value}.")
    x_values = np.array([pair[variable_index] for pair in pairs])
    ratios = {condition: [] for condition in condition_names(APPENDIX_DYNAMICS)}
    for pair in pairs:
        pair_run = load_run(
            OUTPUT_ROOT, experiment=experiment_name, dynamics=APPENDIX_DYNAMICS,
            pair=pair, snapshot=suite.snapshot,
        )
        for condition, case in pair_run.cases.items():
            t_s, t_r, ratio = learning_times(case.time, case.state, *pair, delta=ratio_delta)
            if not np.isfinite([t_s, t_r, ratio]).all() or min(t_s, t_r, ratio) <= 0:
                raise ValueError(f"Invalid or unreached learning times: {experiment_name}, {pair}, {condition}")
            ratios[condition].append(ratio)
    fig_ratio, ax_ratio = plt.subplots(figsize=(9, 7), constrained_layout=True)
    for condition, values in ratios.items():
        ax_ratio.loglog(
            x_values, values, "o-", color=flow_colors[condition],
            label=condition_label(condition), linewidth=2, markersize=6,
        )
    ax_ratio.axhline(1, color="0.65", linestyle=":", linewidth=1)
    ax_ratio.set_xticks(x_values, labels=[str(value) for value in x_values])
    ax_ratio.set(
        xlabel=f"${varying_axis}$", ylabel=r"$T_S^\delta / T_R^\delta$",
        title=f"{fixed_axis} = {fixed_value}, accuracy = {100 * (1 - ratio_delta):g}%",
    )
    ax_ratio.legend(frameon=False)
    pdf_path = APPENDIX_FIGURE_DIR / f"{experiment_name}_delta{ratio_delta:g}.pdf"
    fig_ratio.savefig(pdf_path, format="pdf", bbox_inches="tight")
    print(f"Saved PDF: {pdf_path}")
    ratio_results[experiment_name] = (x_values, ratios)
    ratio_figures[experiment_name] = (fig_ratio, ax_ratio)
    plt.show()


In [ ]:
precision_run = load_run(
    OUTPUT_ROOT, experiment="appendix_precision", dynamics=APPENDIX_DYNAMICS
)
precision_problem = next(iter(precision_run.cases.values())).config.problem
precision_s, precision_r = precision_problem.subjects, precision_problem.relations
# Evaluation errors are notebook choices, independent of the single stopping delta.
ERROR_GRID = np.unique(np.r_[np.logspace(-6, -1, 26), 0.5])
precision_times = {}
precision_figures = {}
for condition, case in precision_run.cases.items():
    times = np.array([
        learning_times(case.time, case.state, precision_s, precision_r, delta=delta)[:2]
        for delta in ERROR_GRID
    ])
    if not np.isfinite(times).all() or np.any(times <= 0):
        raise ValueError("The precision trajectory must reach every positive learning-time target.")
    precision_times[condition] = times
    fig_times, ax_times = plt.subplots(figsize=(9, 7), constrained_layout=True)
    for component, style, column in (("S", "-", 0), ("R", "--", 1)):
        ax_times.loglog(
            ERROR_GRID, times[:, column], linestyle=style, color=("tab:blue", "tab:orange")[column],
            label=rf"$T_{component}^\delta$", linewidth=2.5,
        )
    ax_times.set(
        xlabel=r"Target error $\delta$", ylabel=r"Learning time $T^\delta$",
        title=f"{condition_label(condition)}: S = {precision_s}, R = {precision_r}",
    )
    if condition == "spectral":
        # Format both decades and intermediate ticks as plain numbers.
        ax_times.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
        ax_times.yaxis.set_minor_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
    ax_times.legend(frameon=False)
    pdf_path = APPENDIX_FIGURE_DIR / (
        f"{condition}_learning_time_vs_error_S{precision_s}_R{precision_r}.pdf"
    )
    fig_times.savefig(pdf_path, format="pdf", bbox_inches="tight")
    print(f"Saved PDF: {pdf_path}")
    precision_figures[condition] = (fig_times, ax_times)
    plt.show()
